# 📊 Notebook 1: Exploratory Data Analysis
## RiskForge

**Objectives:**
1. Understand dataset structure and feature distributions
2. Identify class imbalance and address strategy
3. Analyze feature correlations with target variable
4. Detect outliers and missing values
5. Generate business insights from data patterns

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:.3f}'.format)

import sys
sys.path.append('..')
from src.data.generate_data import generate_credit_data

# Generate or load data
df = pd.read_csv('../data/raw/train.csv')
print(f'Dataset shape: {df.shape}')
print(f'Default rate: {df["default"].mean():.2%}')
df.head()

In [ ]:
# ── Class Imbalance Analysis ──────────────────────────────────────────────────
fig = px.pie(
    df, names=df['default'].map({0: 'Non-Default', 1: 'Default'}),
    title='Target Variable Distribution',
    color_discrete_map={'Non-Default': '#27ae60', 'Default': '#e74c3c'}
)
fig.show()

print("Class imbalance ratio:", df['default'].value_counts().to_dict())
print("This ~80/20 split requires SMOTE or class_weight='balanced' during training.")

In [ ]:
# ── Feature Distribution by Default Status ────────────────────────────────────
numeric_features = ['credit_score', 'annual_income', 'loan_amount', 
                    'debt_to_income_ratio', 'revolving_utilization', 'employment_years']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(numeric_features):
    for label, color in [(0, '#27ae60'), (1, '#e74c3c')]:
        axes[i].hist(
            df[df['default'] == label][feat].dropna(),
            alpha=0.6, bins=30, color=color,
            label='Non-Default' if label == 0 else 'Default'
        )
    axes[i].set_title(feat.replace('_', ' ').title(), fontweight='bold')
    axes[i].legend()

plt.suptitle('Feature Distributions by Default Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────────────────────
corr = df[numeric_features + ['default']].corr()

fig = px.imshow(
    corr, text_auto=True, aspect='auto',
    color_continuous_scale='RdBu_r',
    title='Feature Correlation Matrix'
)
fig.show()

# Correlation with target
target_corr = corr['default'].sort_values(key=abs, ascending=False).drop('default')
print('\nTop correlations with default:')
print(target_corr.to_string())

In [ ]:
# ── Default Rate by Credit Score Bucket ──────────────────────────────────────
df['credit_score_bucket'] = pd.cut(df['credit_score'], 
                                    bins=[300, 579, 669, 739, 799, 850],
                                    labels=['Poor', 'Fair', 'Good', 'Very Good', 'Exceptional'])

default_by_bucket = df.groupby('credit_score_bucket', observed=True)['default'].agg(['mean', 'count'])
default_by_bucket.columns = ['Default Rate', 'Count']

fig = px.bar(
    default_by_bucket.reset_index(),
    x='credit_score_bucket', y='Default Rate',
    text='Default Rate',
    title='Default Rate by Credit Score Tier',
    color='Default Rate',
    color_continuous_scale='RdYlGn_r'
)
fig.update_traces(texttemplate='%{text:.1%}')
fig.show()

print(default_by_bucket)

In [ ]:
# ── Missing Value Analysis ────────────────────────────────────────────────────
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_report = missing_report[missing_report['Missing Count'] > 0]

print('Missing Value Report:')
print(missing_report)
print('\nStrategy: months_since_last_delinq is missing when applicant has no delinquencies.')
print('Impute with 99 (business logic: no delinquency = best possible)')

## 📋 EDA Summary

| Finding | Implication |
|---|---|
| 20% default rate | Use SMOTE + class_weight for training |
| Credit score most correlated with default | Include in top features |
| Income is right-skewed | Apply log transform |
| months_since_delinq has 70% missingness | Business-logic imputation |
| DTI and revolving_util interact | Create interaction feature |
